In [1]:
from ddgs import DDGS
from IPython.display import display, Image

### Trying out the structure of the data

In [2]:
def search_images(keyword, max_results = 10, show_pictures=False, pictures_shown = 3):
    with DDGS() as ddgs:
        results = ddgs.images(keyword, max_results = max_results)
        print(f"Fetched {len(results)} images")

        # Returns a list of strings (strings being the URLs)
        result_urls = [result_dict["image"] for result_dict in results]

        # In case the user wants to see the images
        if show_pictures:
            print(f"Here's the first {pictures_shown} of them:")
            for im_url in result_urls[0:min(pictures_shown, len(result_urls))]:
                display(Image(url = im_url, width = 200))
        return result_urls

In [3]:
image_urls = search_images("leaf", 10, show_pictures = True)

Fetched 10 images
Here's the first 3 of them:


### Importing the data

In [4]:
import os
import requests
from urllib.parse import urlparse
import warnings

def download_image(url, folder, custom_name=None, verbose=True):
    # Create the folder if it doesn't exist
    os.makedirs(folder, exist_ok=True)

    # Get the filename from the URL or use the custom name
    if custom_name:
        filename = custom_name
    else:
        filename = os.path.basename(urlparse(url).path)
        if not filename:
            filename = 'image.jpg'  # Default filename if none is found in the URL

    # Ensure the filename has an extension
    if not os.path.splitext(filename)[1]:
        filename += '.jpg'

    filepath = os.path.join(folder, filename)

    # If the file already exists, append a number to make it unique
    base, extension = os.path.splitext(filepath)
    counter = 1
    while os.path.exists(filepath):
        filepath = f"{base}_{counter}{extension}"
        counter += 1

    try:
        # Send a GET request to the URL with a timeout of 10 seconds
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raises an HTTPError for bad responses

        # Check if the content type is an image
        content_type = response.headers.get('content-type', '')
        if not content_type.startswith('image'):
            if verbose:
                warnings.warn(f"The URL does not point to an image. Content-Type: {content_type}")
            return False

        # Write the image content to the file
        with open(filepath, 'wb') as f:
            f.write(response.content)

        if verbose:
            print(f"Image successfully downloaded: {filepath}")
        return filepath


    except requests.exceptions.Timeout:
        if verbose: 
            warnings.warn(f"Download timed out for URL: {url}")
    except requests.exceptions.HTTPError as e:
        if verbose: 
            warnings.warn(f"HTTP error occurred: {e}")
    except requests.exceptions.RequestException as e:
        if verbose: 
            warnings.warn(f"An error occurred while downloading the image: {e}")
    except IOError as e:
        if verbose: 
            warnings.warn(f"An error occurred while writing the file: {e}")

    return None

In [5]:
import random 
import shutil

def train_test_divide(filepaths: list[str], titles_for_subfolders: list[str], index: int, split_ratio = 0.75):
    # We shuffle the data just in case
    random.shuffle(filepaths)

    # Next we split the data into two parts
    split_index = int(len(filepaths) * split_ratio)

    train_files = filepaths[:split_index]
    test_files = filepaths[split_index:]

    train_dir = f"../../dataset/train/{titles_for_subfolders[index]}"
    test_dir = f"../../dataset/test/{titles_for_subfolders[index]}"
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    for file in train_files:
        shutil.move(file, train_dir)
    for file in test_files:
        shutil.move(file, test_dir)
    

In [6]:
from tqdm.notebook import tqdm

def search_and_save_images(keywords: list, titles_for_subfolders: list, max_results: int, batch_num: int, verbose = False):
    if type(keywords) != list or type(titles_for_subfolders) != list:
        raise TypeError("Please use lists for inputting information")
    if len(keywords) != len(titles_for_subfolders):
        raise ValueError("Incorrect number of elements in lists")

    for i, keyword in enumerate(keywords):
        # Search for the images using the keyword
        image_urls = search_images(keyword, max_results)

        successful_filepaths = []
        temp_folder_dir = f"../../dataset/temp_{titles_for_subfolders[i]}"
        os.makedirs(temp_folder_dir, exist_ok=True)

        for j, im_url in enumerate(tqdm(image_urls)):
            filepath = download_image(im_url, temp_folder_dir, f'b{batch_num}_image{j}.jpg', verbose=verbose)
            if filepath:
                successful_filepaths.append(filepath)
        
        train_test_divide(successful_filepaths, titles_for_subfolders, i)
        os.rmdir(temp_folder_dir)
        
        print(f"Successfully downloaded {len(successful_filepaths)} images out of {len(image_urls)} possible ones for the keyword '{keyword}'")


In [8]:
keyword_list_1 = ["ginkgo leaf isolated on white", 
                  "monstera leaf close up", 
                  "oak leaf isolated", 
                  "red maple leaf close up", 
                  "calathea orbifolia leaf isolated on white"
                  ]

keyword_list_2 = ["ginkgo biloba leaf single close up", 
                  "monstera plant leaf isolated", 
                  "oak leaf single on white", 
                  "green maple leaf close up", 
                  "calathea orbifolia leaf single close up"
                  ]

keyword_list_3 = ["ginkgo tree leaf macro detail", 
                  "monstera leaf isolated on white", 
                  "oak leaf green white close up", 
                  "maple leaf close up white", 
                  "calathea orbifolia leaf detail close up"
                  ]

titles_for_subfolders_list = ["ginkgo_leaves", 
                              "monstera_leaves", 
                              "oak_leaves", 
                              "maple_leaves",
                              "calathea_leaves"
                              ]

search_and_save_images(
    keywords = keyword_list_1,
    titles_for_subfolders = titles_for_subfolders_list,
    batch_num= 1,
    max_results = 100
)

search_and_save_images(
    keywords = keyword_list_2,
    titles_for_subfolders = titles_for_subfolders_list,
    batch_num = 2,
    max_results = 100
)

search_and_save_images(
    keywords = keyword_list_3,
    titles_for_subfolders = titles_for_subfolders_list,
    batch_num = 3,
    max_results = 100
)

Fetched 100 images


  0%|          | 0/100 [00:00<?, ?it/s]

Successfully downloaded 96 images out of 100 possible ones for the keyword 'ginkgo leaf isolated on white'
Fetched 100 images


  0%|          | 0/100 [00:00<?, ?it/s]

Successfully downloaded 98 images out of 100 possible ones for the keyword 'monstera leaf close up'
Fetched 100 images


...

Successfully downloaded 92 images out of 100 possible ones for the keyword 'oak leaf green white close up'
Fetched 100 images


  0%|          | 0/100 [00:00<?, ?it/s]

Successfully downloaded 94 images out of 100 possible ones for the keyword 'maple leaf close up white'
Fetched 89 images


  0%|          | 0/89 [00:00<?, ?it/s]

Successfully downloaded 84 images out of 89 possible ones for the keyword 'calathea orbifolia leaf detail close up'
